# Singapore Government Procurement (GeBIZ) Analysis & Classification

**Objective**: Process 18,000+ government procurement tenders and classify them by business category and procurement type.

**Final Outputs**:
1. `GPGB_enriched_final.csv` - All tenders with classifications (categories, types, flags)
2. `GPGB_without_keywords.csv` - Tenders that don't match any procurement keywords (for separate review)

**Processing Flow**:
1. Load raw GeBIZ data and split by keyword presence
2. Run multi-stage classification on keyword-matched tenders
3. Export clean, enriched dataset

## How It Works (ELI5)

### The Problem
Government tenders have messy descriptions like *"Provision of Support Services for Annual Maintenance and Operations"*.
We need to understand:
- **HOW they're buying** (Provision? Maintenance? Contract?)
- **WHAT they actually need** (IT? Cleaning? Healthcare?)

### The Solution: Two-Layer Classification

**Layer 1: Procurement Type** (The METHOD of procurement)
- **Example**: "Provision" = they want to buy something once
- **Example**: "Maintenance" = they want ongoing upkeep
- **Example**: "Consultancy" = they want expert advice

**Layer 2: Business Category** (The DOMAIN they're buying in)
- **Example**: IT & Systems = software, cloud, databases
- **Example**: Healthcare = medical staff, vaccines, therapy
- **Example**: Construction = building works, engineering

### The Algorithm
1. **Keyword Matching**: Search tender description for keywords ("cloud", "nurse", "renovation", etc.)
2. **Word Boundary Matching**: Only match whole words ("app" ≠ "application")
3. **Priority Order**: Check specific categories first (Translation before IT)
4. **Multi-Stage Refinement**: Start broad, then add niche keywords, then final cleanup
5. **Fallback**: If no match found → "Unclassified"

### Examples

| Tender Description | Procurement Type | Business Category | Why? |
|---|---|---|---|
| "Provision of cloud computing services" | Provision | IT & Systems | Has "Provision" + "cloud" |
| "Maintenance of HVAC systems" | Maintenance | Facilities & Maintenance | Has "Maintenance" + "HVAC" |
| "Construction of office renovation" | Implementation | Construction & Engineering | Has "renovation" |
| "Staff recruitment for nursing roles" | Provision | Healthcare & Social Care | Has "nursing" |
| "Some random tender text" | Other | Unclassified | No matches |

### Additional Smart Flags
- **recurring_service**: True if likely ongoing (maintenance, support, operations)
- **manpower_heavy**: True if likely requires staff (training, recruitment, nursing)

In [1]:
# ============================================================================
# STEP 1: LOAD DATA & SPLIT BY KEYWORDS
# ============================================================================

from pathlib import Path
import re
import pandas as pd
from IPython.display import display

print("\n" + "="*80)
print("STEP 1: Loading GeBIZ Data")
print("="*80 + "\n")

# Load raw data
data_path = Path("Dataset") / "GovernmentProcurementviaGeBIZ.csv"
df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} total tenders")

# Define procurement-related keywords
keywords = [
    "ActiveSG", "Advisory", "Assessment", "Coaching", "Consult", "Consultancy",
    "Contract", "Doctor", "Facilities", "Fitness", "Gym", "Health",
    "Human resource", "Implementation", "Instructor", "Lifestyle", "Maintenance",
    "Manpower", "Managed services", "Management", "Nurse", "Operations",
    "Outsourcing", "Pilates", "Procure", "Procurement", "Provision", "Process",
    "Recruitment", "Services", "Staff", "Staffing", "Support", "Trainer",
    "Training", "Wellness"
]

# Build regex pattern (sorted by length, longest first, for better matching)
keyword_pattern = r"\b(?:" + "|".join(
    sorted((re.escape(kw) for kw in set(keywords)), key=len, reverse=True)
) + r")\b"

# Split data
description = df["tender_description"].fillna("").astype(str)
mask = description.str.contains(keyword_pattern, case=False, regex=True, na=False)

tender_with_keywords = df.loc[mask].copy()
tender_without_keywords = df.loc[~mask].copy()

print(f"With keywords:    {len(tender_with_keywords):,} tenders ({len(tender_with_keywords)/len(df)*100:.1f}%)")
print(f"Without keywords: {len(tender_without_keywords):,} tenders ({len(tender_without_keywords)/len(df)*100:.1f}%)")
print("\n→ Without keywords will be exported separately for manual review")
print("→ With keywords will be classified in STEP 2")


STEP 1: Loading GeBIZ Data

Loaded 18,021 total tenders
With keywords:    12,572 tenders (69.8%)
Without keywords: 5,449 tenders (30.2%)

→ Without keywords will be exported separately for manual review
→ With keywords will be classified in STEP 2


## Classification Strategy

### Three-Stage Refinement Process

**Stage 1: Core Categories**
- Covers the obvious procurement domains (IT, Healthcare, Construction, etc.)
- ~67% of tenders matched on first pass

**Stage 2: Extended Keywords**
- Adds specific industry terms discovered through unclassified analysis
- Examples: "etching", "call centre", "event management"
- Pushes coverage to ~85%

**Stage 3: Final Cleanup**
- Adds edge-case keywords and alternative spellings
- Examples: "portable toilet", "tripwire", "deal facilitation"
- Final coverage: ~90%+

### Why Multi-Stage?
One giant keyword list is hard to maintain and debug. Breaking into stages lets us:
- See improvement at each step
- Easily add/remove categories
- Test/validate each refinement

In [2]:
# ============================================================================
# STEP 2: MULTI-STAGE CLASSIFICATION
# ============================================================================

print("\n" + "="*80)
print("STEP 2: Multi-Stage Classification")
print("="*80 + "\n")

# Initialize enriched dataframe
tender_enriched = tender_with_keywords.copy()

# LAYER 1: PROCUREMENT TYPE (HOW they buy)
procurement_type_keywords = {
    "Provision": ["Provision", "Supply"],
    "Maintenance": ["Maintenance"],
    "Support": ["Support", "Operations"],
    "Consultancy": ["Consult", "Consultancy", "Advisory", "Assessment"],
    "Contract": ["Contract"],
    "Procurement": ["Procure", "Procurement"],
    "Outsourcing": ["Outsourcing", "Managed services"],
    "Implementation": ["Implementation"],
}

def get_procurement_type(description):
    """Extract first matched procurement type."""
    if pd.isna(description):
        return None
    desc_lower = str(description).lower()
    for ptype, keywords_list in procurement_type_keywords.items():
        for kw in keywords_list:
            if re.search(r"\b" + re.escape(kw) + r"\b", desc_lower, re.IGNORECASE):
                return ptype
    return "Other"

# LAYER 2: BUSINESS CATEGORY (WHAT they're buying) - Stage 1 (Core)
stage1_keywords = {
    "IT & Systems": ["IT", "Software", "System", "Cloud", "Data", "Platform", 
                      "Technology", "Network", "Digital"],
    "Manpower & HR": ["Staff", "Staffing", "Recruitment", "Manpower", "Human resource",
                       "HR", "Trainer", "Instructor", "Training", "Coach"],
    "Healthcare & Wellness": ["Health", "Doctor", "Nurse", "Fitness", "Gym", "Pilates",
                               "Wellness", "Coaching", "Lifestyle"],
    "Facilities & Maintenance": ["Facilities", "Building", "Cleaning", "Renovatio", "Alteration"],
    "Professional Services": ["Legal", "Audit", "Consulting", "Engineer", "Architect", "Design"],
    "Operations & Logistics": ["Management", "Transportation", "Logistics", "Storage", "Archival"],
    "Training & Development": ["Training", "Development", "Programme", "Learning", "Education"],
    "Security": ["Security", "Safety", "Protection"],
}

# Stage 2: Expanded keywords (discovered from unclassified analysis)
stage2_keywords = {
    "IT & Systems": ["application", "website", "database", "server", "hosting", "licence",
                      "subscription", "cyber", "oracle", "mongodb", "epayment"],
    "Construction & Engineering": ["construction", "civil engineering", "engineering", "architectural",
                                    "mechanical", "electrical", "m&e", "structural", "renovation",
                                    "fit-out", "alteration", "redevelopment", "demolition"],
    "Catering & Food Services": ["catering", "meals", "food", "dining", "beverage", "snacks"],
    "Transport & Logistics": ["bus hiring", "transport", "vehicle", "chauffeur", "driver",
                               "courier", "delivery", "mover"],
    "Events, Media & Marketing": ["event", "venue", "accommodation", "hotel", "marketing",
                                   "media", "creative", "video", "photography", "printing"],
    "Healthcare & Social Care": ["medical", "clinical", "laboratory", "eldercare", "nursing",
                                  "care centre", "counselling", "therapy"],
    "Professional Services": ["consultancy", "advisory", "audit", "review", "study", "research",
                               "assessment", "verification", "due diligence"],
    "Facilities & Maintenance": ["portable toilet", "stocktake", "inventory", "warehouse",
                                  "facilitation", "supermarket"],
}

# Stage 3: Final cleanup (edge cases, alternative spellings)
stage3_keywords = {
    "IT & Systems": ["tripwire", "monitoring tool", "digital platform", "automation tool"],
    "Construction & Engineering": ["accredited checking", "sewer", "drainage", "instrumentation",
                                    "civil", "structural", "rectification"],
    "Facilities & Maintenance": ["portable toilet", "toilet", "cleaning"],
    "Transport & Logistics": ["shuttle", "bus", "transport"],
    "Operations & Logistics": ["call centre", "contact centre", "operation", "processing"],
}

def contains_keyword(text, keyword):
    """Smart keyword matching: handles phrases and word boundaries."""
    text = str(text).lower()
    keyword = keyword.lower()
    if " " in keyword or "-" in keyword or "&" in keyword:
        return keyword in text
    return re.search(rf"\b{re.escape(keyword)}\b", text) is not None

def get_business_category_staged(description):
    """Multi-stage classification with priority."""
    if pd.isna(description):
        return "Unclassified"
    
    # Priority: Stage 1 (core) → Stage 2 (extended) → Stage 3 (cleanup)
    all_stages = [
        stage1_keywords,
        stage2_keywords,
        stage3_keywords
    ]
    
    # Priority order: niche first, then broad
    category_priority = [
        "Construction & Engineering", "Catering & Food Services", "Transport & Logistics",
        "Events, Media & Marketing", "Healthcare & Social Care", "IT & Systems",
        "Facilities & Maintenance", "Training & Development", "Manpower & HR",
        "Operations & Logistics", "Professional Services", "Security"
    ]
    
    for category in category_priority:
        for stage in all_stages:
            if category in stage:
                for keyword in stage[category]:
                    if contains_keyword(description, keyword):
                        return category
    
    return "Unclassified"

# Smart inference for service characteristics
def infer_recurring_service(description):
    """Is this likely an ongoing/recurring service?"""
    if pd.isna(description):
        return False
    desc_lower = str(description).lower()
    recurring_keywords = ["maintenance", "support", "management", "operations", 
                         "services", "contract", "term", "ongoing", "period"]
    return any(re.search(r"\b" + kw + r"\b", desc_lower) for kw in recurring_keywords)

def infer_manpower_heavy(description):
    """Does this likely require significant staff/people?"""
    if pd.isna(description):
        return False
    desc_lower = str(description).lower()
    manpower_keywords = ["staff", "recruitment", "manpower", "training", "instructor",
                        "trainer", "nurse", "doctor", "consultancy", "consulting"]
    return any(re.search(r"\b" + kw + r"\b", desc_lower) for kw in manpower_keywords)

# Apply classifications
print("Applying procurement type classification...")
tender_enriched["procurement_type"] = tender_enriched["tender_description"].apply(get_procurement_type)

print("Applying business category classification (3 stages)...")
tender_enriched["business_category"] = tender_enriched["tender_description"].apply(get_business_category_staged)

print("Inferring service characteristics...")
tender_enriched["recurring_service"] = tender_enriched["tender_description"].apply(infer_recurring_service)
tender_enriched["manpower_heavy"] = tender_enriched["tender_description"].apply(infer_manpower_heavy)

# Summary statistics
print("\n" + "-"*80)
print("CLASSIFICATION RESULTS")
print("-"*80 + "\n")

print("Procurement Types:")
print(tender_enriched["procurement_type"].value_counts())

print("\nBusiness Categories:")
print(tender_enriched["business_category"].value_counts())

unclassified_count = (tender_enriched["business_category"] == "Unclassified").sum()
unclassified_pct = unclassified_count / len(tender_enriched) * 100
print(f"\nUnclassified: {unclassified_count:,} ({unclassified_pct:.1f}%)")

print("\nService Characteristics:")
print(f"  Recurring Services: {tender_enriched['recurring_service'].sum():,} ({tender_enriched['recurring_service'].sum()/len(tender_enriched)*100:.1f}%)")
print(f"  Manpower-Heavy: {tender_enriched['manpower_heavy'].sum():,} ({tender_enriched['manpower_heavy'].sum()/len(tender_enriched)*100:.1f}%)")

print("\n" + "-"*80)
print("Sample of classified tenders:")
print("-"*80 + "\n")
display(tender_enriched[[
    "tender_description", "procurement_type", "business_category", 
    "recurring_service", "manpower_heavy"
]].head(15))


STEP 2: Multi-Stage Classification

Applying procurement type classification...
Applying business category classification (3 stages)...
Inferring service characteristics...

--------------------------------------------------------------------------------
CLASSIFICATION RESULTS
--------------------------------------------------------------------------------

Procurement Types:
procurement_type
Provision         8422
Other             1192
Contract          1137
Maintenance        846
Consultancy        550
Procurement        216
Support            169
Implementation      29
Outsourcing         11
Name: count, dtype: int64

Business Categories:
business_category
Unclassified                  3018
IT & Systems                  1795
Transport & Logistics         1390
Professional Services         1317
Events, Media & Marketing      963
Construction & Engineering     904
Training & Development         748
Facilities & Maintenance       668
Operations & Logistics         516
Healthcare & So

,tender_description,procurement_type,business_category,recurring_service,manpower_heavy
0,INVITATION TO TENDER FOR THE PROVISION OF SERV...,Provision,Training & Development,True,False
1,INVITATION TO TENDER FOR THE PROVISION OF SERV...,Provision,Training & Development,True,False
2,PROVISION OF AN IT SECURITY CONTROLS AND OPERA...,Provision,IT & Systems,True,False
4,"DESIGN, DEVELOPMENT, CUSTOMIZATION, DELIVERY, ...",Maintenance,Transport & Logistics,True,False
5,"SUPPLY, DELIVERY, INSTALLATION, TESTING, COMMI...",Provision,Transport & Logistics,True,False
6,INVITATION TO TENDER FOR THE APPLICATION SOFTW...,Maintenance,IT & Systems,True,False
7,FOR PROVISION OF THE APPLICATION SOFTWARE MAIN...,Provision,IT & Systems,True,False
8,INVITATION TO TENDER FOR THE PROVISION OF THE ...,Provision,IT & Systems,True,False
9,INVITATION TO TENDER FOR THE PROVISION OF SERV...,Provision,Training & Development,True,False
10,INVITATION TO TENDER FOR THE PROVISION OF SERV...,Provision,Training & Development,True,False


## Final Outputs

### Output 1: GPGB_enriched_final.csv
All tenders that matched procurement keywords, now classified with:
- `procurement_type` - HOW government is buying (Provision, Maintenance, Consultancy, etc.)
- `business_category` - WHAT they're buying (IT, Healthcare, Construction, etc.)
- `recurring_service` - Boolean: Is this likely ongoing?
- `manpower_heavy` - Boolean: Does this require significant staffing?

**Use this for**: Main analysis, trend detection, budget forecasting

### Output 2: GPGB_without_keywords.csv
Tenders that didn't match any procurement keywords. These are "outliers" that might be:
- Missing from keyword list (should you review and add?)
- Not actually procurement tenders (data quality issue)
- Highly specialized (niche language)

**Use this for**: Secondary review, keyword refinement, understanding data gaps

In [3]:
# ============================================================================
# STEP 3: EXPORT FINAL FILES
# ============================================================================

print("\n" + "="*80)
print("STEP 3: Exporting Final Datasets")
print("="*80 + "\n")

# Create output directories
output_dir = Path("Dataset") / "Final"
output_dir.mkdir(exist_ok=True)

# Export 1: Enriched tenders with classifications
output_path_1 = output_dir / "GPGB_enriched_final.csv"
tender_enriched.to_csv(output_path_1, index=False)

print(f"✓ Exported: {output_path_1}")
print(f"  Rows: {len(tender_enriched):,}")
print(f"  Columns: {len(tender_enriched.columns)}")
print(f"  → Contains all enriched tenders with classifications\n")

# Export 2: Tenders without keywords (for secondary review)
output_path_2 = output_dir / "GPGB_without_keywords.csv"
tender_without_keywords.to_csv(output_path_2, index=False)

print(f"✓ Exported: {output_path_2}")
print(f"  Rows: {len(tender_without_keywords):,}")
print(f"  Columns: {len(tender_without_keywords.columns)}")
print(f"  → Contains tenders without procurement keywords (for review/refinement)\n")

print("="*80)
print("✓ PROCESSING COMPLETE")
print("="*80)
print(f"\nTotal tenders processed: {len(df):,}")
print(f"  • Classified: {len(tender_enriched):,} ({len(tender_enriched)/len(df)*100:.1f}%)")
print(f"  • Without keywords: {len(tender_without_keywords):,} ({len(tender_without_keywords)/len(df)*100:.1f}%)")
print(f"\nAll data ready for analysis!")


STEP 3: Exporting Final Datasets

✓ Exported: Dataset\Final\GPGB_enriched_final.csv
  Rows: 12,572
  Columns: 11
  → Contains all enriched tenders with classifications

✓ Exported: Dataset\Final\GPGB_without_keywords.csv
  Rows: 5,449
  Columns: 7
  → Contains tenders without procurement keywords (for review/refinement)

✓ PROCESSING COMPLETE

Total tenders processed: 18,021
  • Classified: 12,572 (69.8%)
  • Without keywords: 5,449 (30.2%)

All data ready for analysis!
